## Pursuit evasion game

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML
from matplotlib import transforms

import hj_reachability as hj
from hj_reachability.systems.relative_vehicle_6d import RelativeVehicle6D

ROOT = Path.cwd().parent
BRT_PATH = ROOT / "results" / "brt" / "dce.npz"

RECOVERY_HORIZON = 0.6
RECOVERY_DT = 0.2
RECOVERY_YAW_SAMPLES = 4
RECOVERY_ACCELERATION_SAMPLES = 4

saved_data = np.load(BRT_PATH, allow_pickle=True)
metadata = json.loads(
    str(saved_data["metadata_json"].item())
)

grid_lo = np.asarray(
    saved_data["grid_lo"],
    dtype=float,
)

grid_hi = np.asarray(
    saved_data["grid_hi"],
    dtype=float,
)

grid_shape = tuple(
    int(value)
    for value in saved_data["grid_shape"]
)

periodic_dims = tuple(
    int(value)
    for value in saved_data["periodic_dims"]
)

grid = hj.Grid.from_lattice_parameters_and_boundary_conditions(
    domain=hj.sets.Box(
        lo=jnp.asarray(grid_lo),
        hi=jnp.asarray(grid_hi),
    ),
    shape=grid_shape,
    periodic_dims=periodic_dims,
)

dynamics = RelativeVehicle6D(
    **metadata["dynamics"]["parameters"]
)

values = jnp.asarray(saved_data["BRT"])
grad_values = jnp.asarray(saved_data["gradients"])

print(f"Loaded: {BRT_PATH}")
print(f"BRT shape: {values.shape}")
print(f"Gradient shape: {grad_values.shape}")
print(f"Grid bounds:\n{grid_lo}\n{grid_hi}")

def relative_state(x):
    """
    Absolute joint state:

    x = [
        x_E, y_E, psi_E, delta_E, v_E,
        x_H, y_H, psi_H, v_H
    ]

    Relative state:

    x_R = [
        x_rel, y_rel, theta_rel,
        v_H, delta_E, v_E
    ]
    """
    (
        x_e,
        y_e,
        psi_e,
        delta_e,
        v_e,
        x_h,
        y_h,
        psi_h,
        v_h,
    ) = x

    rot_matrix = jnp.array(
        [
            [jnp.cos(psi_e), jnp.sin(psi_e)],
            [-jnp.sin(psi_e), jnp.cos(psi_e)],
        ]
    )

    relative_position = (
        rot_matrix
        @ jnp.array(
            [
                x_h - x_e,
                y_h - y_e,
            ]
        )
    )

    theta_rel = psi_h - psi_e

    # Generalizzazione del:
    # jnp.mod(qb - qa, 2*pi)
    # usato nel notebook Stanford.
    if 2 in periodic_dims:
        theta_period = grid_hi[2] - grid_lo[2]

        theta_rel = (
            grid_lo[2]
            + jnp.mod(
                theta_rel - grid_lo[2],
                theta_period,
            )
        )

    return jnp.array(
        [
            relative_position[0],
            relative_position[1],
            theta_rel,
            v_h,
            delta_e,
            v_e,
        ]
    )

def joint_dynamics(
    x,
    u,
    relative_dynamics=dynamics,
):
    """
    u = [
        steering_rate_E,
        acceleration_E,
        yaw_rate_H,
        acceleration_H
    ]
    """
    (
        x_e,
        y_e,
        psi_e,
        delta_e,
        v_e,
        x_h,
        y_h,
        psi_h,
        v_h,
    ) = x

    steering_rate_e = u[0]
    acceleration_e = u[1]

    yaw_rate_h = u[2]
    acceleration_h = u[3]

    lf = relative_dynamics.lf
    lr = relative_dynamics.lr

    beta_e = jnp.arctan(
        lr
        / (lr + lf)
        * jnp.tan(delta_e)
    )

    yaw_rate_e = (
        v_e
        * jnp.cos(beta_e)
        / (lr + lf)
        * jnp.tan(delta_e)
    )

    return jnp.array(
        [
            # Ego vehicle
            v_e * jnp.cos(psi_e + beta_e),
            v_e * jnp.sin(psi_e + beta_e),
            yaw_rate_e,
            steering_rate_e,
            acceleration_e,

            # Human vehicle
            v_h * jnp.cos(psi_h),
            v_h * jnp.sin(psi_h),
            yaw_rate_h,
            acceleration_h,
        ]
    )

@jax.jit
def joint_step(joint_state, dt, t):
    state = relative_state(joint_state)

    value = grid.interpolate(
        values,
        state,
    )

    grad_value = grid.interpolate(
        grad_values,
        state,
    )

    ego_control, human_disturbance = (
        dynamics.optimal_control_and_disturbance(
            state,
            t * dt,
            grad_value,
        )
    )

    ego_control = jnp.asarray(ego_control)
    human_disturbance = jnp.asarray(
        human_disturbance
    )

    # Relative state:
    # [x_rel, y_rel, theta_rel, v_H, delta_E, v_E]

    v_h = state[3]
    delta_e = state[4]
    v_e = state[5]

    v_h_min = grid_lo[3]
    v_h_max = grid_hi[3]

    delta_e_min = grid_lo[4]
    delta_e_max = grid_hi[4]

    v_e_min = grid_lo[5]
    v_e_max = grid_hi[5]

    # ========================================================
    # ADMISSIBLE INPUTS FOR THE NEXT EULER STEP
    # ========================================================

    # Human acceleration:
    # v_H_next = v_H + a_H * dt
    human_acceleration_min = (
        v_h_min - v_h
    ) / dt

    human_acceleration_max = (
        v_h_max - v_h
    ) / dt

    # Ego steering rate:
    # delta_E_next = delta_E + delta_dot_E * dt
    ego_steering_rate_min = (
        delta_e_min - delta_e
    ) / dt

    ego_steering_rate_max = (
        delta_e_max - delta_e
    ) / dt

    # Ego acceleration:
    # v_E_next = v_E + a_E * dt
    ego_acceleration_min = (
        v_e_min - v_e
    ) / dt

    ego_acceleration_max = (
        v_e_max - v_e
    ) / dt

    # ========================================================
    # CLIP OPTIMAL HJ INPUTS
    # ========================================================

    # ego_control:
    # [steering_rate_E, acceleration_E]

    ego_control = ego_control.at[0].set(
        jnp.clip(
            ego_control[0],
            ego_steering_rate_min,
            ego_steering_rate_max,
        )
    )

    ego_control = ego_control.at[1].set(
        jnp.clip(
            ego_control[1],
            ego_acceleration_min,
            ego_acceleration_max,
        )
    )

    # human_disturbance:
    # [yaw_rate_H, acceleration_H]

    human_disturbance = (
        human_disturbance.at[1].set(
            jnp.clip(
                human_disturbance[1],
                human_acceleration_min,
                human_acceleration_max,
            )
        )
    )

    joint_input = jnp.concatenate(
        [
            ego_control,
            human_disturbance,
        ]
    )

    # Explicit Euler integration, as in Stanford.
    next_joint_state = (
        joint_state
        + joint_dynamics(
            joint_state,
            joint_input,
        )
        * dt
    )

    # ========================================================
    # NUMERICAL SAFETY CLIP
    # ========================================================
    #
    # Absolute joint state:
    # [
    #   x_E, y_E, psi_E, delta_E, v_E,
    #   x_H, y_H, psi_H, v_H
    # ]

    next_joint_state = (
        next_joint_state
        .at[3]
        .set(
            jnp.clip(
                next_joint_state[3],
                delta_e_min,
                delta_e_max,
            )
        )
    )

    next_joint_state = (
        next_joint_state
        .at[4]
        .set(
            jnp.clip(
                next_joint_state[4],
                v_e_min,
                v_e_max,
            )
        )
    )

    next_joint_state = (
        next_joint_state
        .at[8]
        .set(
            jnp.clip(
                next_joint_state[8],
                v_h_min,
                v_h_max,
            )
        )
    )

    return (
        next_joint_state,
        value,
    )


def joint_trajectory(
    ego_state,
    human_state,
    dt=1 / 30,
    T=10,
):
    """
    Stanford-style simulation with recovery outside the BRT grid.

    HJ mode:
        control and disturbance are obtained from the BRT gradient.

    Recovery mode:
        - ego brings delta_E to zero;
        - ego returns to the last in-grid velocity;
        - human selects yaw rate and acceleration minimizing
          the center-to-center distance at the end of the
          recovery prediction horizon.
    """
    joint_states = [
        np.concatenate(
            [
                ego_state,
                human_state,
            ]
        )
    ]

    values_history = []

    state_names = (
        "x_rel",
        "y_rel",
        "theta_rel",
        "v_H",
        "delta_E",
        "v_E",
    )

    # Input bounds used during BRT computation.
    ego_input_min = np.asarray(
        dynamics.control_space.lo,
        dtype=float,
    )

    ego_input_max = np.asarray(
        dynamics.control_space.hi,
        dtype=float,
    )

    human_input_min = np.asarray(
        dynamics.disturbance_space.lo,
        dtype=float,
    )

    human_input_max = np.asarray(
        dynamics.disturbance_space.hi,
        dtype=float,
    )

    # Candidate human controls.
    human_yaw_candidates = np.unique(
        np.concatenate(
            [
                np.linspace(
                    human_input_min[0],
                    human_input_max[0],
                    RECOVERY_YAW_SAMPLES,
                ),
                np.array([0.0]),
            ]
        )
    )

    human_acceleration_candidates = np.unique(
        np.concatenate(
            [
                np.linspace(
                    human_input_min[1],
                    human_input_max[1],
                    RECOVERY_ACCELERATION_SAMPLES,
                ),
                np.array([0.0]),
            ]
        )
    )

    recovery_prediction_steps = int(
        np.round(
            RECOVERY_HORIZON
            / RECOVERY_DT
        )
    )

    if recovery_prediction_steps < 1:
        raise ValueError(
            "RECOVERY_HORIZON must be greater than or "
            "equal to RECOVERY_DT."
        )

    recovery_active = False
    recovery_reference_speed = None

    # Last ego speed associated with a valid in-grid state.
    last_valid_ego_speed = float(
        ego_state[4]
    )

    for t in range(int(T / dt)):
        current_time = t * dt

        current_joint_state = np.asarray(
            joint_states[-1],
            dtype=float,
        )

        current_relative_state = np.asarray(
            relative_state(
                jnp.asarray(current_joint_state)
            ),
            dtype=float,
        )

        # ====================================================
        # GRID CHECK
        # ====================================================

        outside_dimensions = []

        for dimension in range(6):
            if dimension in periodic_dims:
                continue

            value = current_relative_state[dimension]

            if (
                value < grid_lo[dimension]
                or value > grid_hi[dimension]
            ):
                outside_dimensions.append(dimension)

        inside_grid = not outside_dimensions

        # Save the last valid ego velocity.
        if inside_grid and not recovery_active:
            last_valid_ego_speed = float(
                current_relative_state[5]
            )

        # ====================================================
        # MODE TRANSITIONS
        # ====================================================

        if not inside_grid and not recovery_active:
            recovery_active = True

            recovery_reference_speed = (
                last_valid_ego_speed
            )

            print("\n" + "=" * 70)
            print("RECOVERY MODE ACTIVATED")
            print("=" * 70)
            print(
                f"Simulation time: "
                f"{current_time:.6f} s"
            )
            print(
                f"Simulation step: {t}"
            )
            print("Outside dimensions:")

            for dimension in outside_dimensions:
                print(
                    f"  {state_names[dimension]} = "
                    f"{current_relative_state[dimension]:.8f}, "
                    f"grid = "
                    f"[{grid_lo[dimension]:.8f}, "
                    f"{grid_hi[dimension]:.8f}]"
                )

            print(
                "Stored ego reference speed: "
                f"{recovery_reference_speed:.8f} m/s"
            )
            print("=" * 70 + "\n")

        elif inside_grid and recovery_active:
            recovery_active = False
            recovery_reference_speed = None

            print("\n" + "=" * 70)
            print("RECOVERY MODE DEACTIVATED")
            print("=" * 70)
            print(
                "State returned inside the grid at "
                f"t = {current_time:.6f} s"
            )
            print("=" * 70 + "\n")

        # ====================================================
        # NORMAL HJ MODE
        # ====================================================

        if not recovery_active:
            next_joint_state, value = joint_step(
                jnp.asarray(current_joint_state),
                dt,
                t,
            )

            joint_states.append(
                np.asarray(
                    next_joint_state,
                    dtype=float,
                )
            )

            values_history.append(
                float(value)
            )

            continue

        # ====================================================
        # RECOVERY MODE: EGO CONTROL
        # ====================================================

        delta_e = current_relative_state[4]
        v_e = current_relative_state[5]

        # Fastest admissible steering-rate command that reaches
        # delta_E = 0 without crossing it.
        ego_steering_rate = np.clip(
            -delta_e / dt,
            ego_input_min[0],
            ego_input_max[0],
        )

        # Acceleration required to return to the velocity stored
        # immediately before leaving the grid.
        ego_acceleration = np.clip(
            (
                recovery_reference_speed
                - v_e
            ) / dt,
            ego_input_min[1],
            ego_input_max[1],
        )

        # Keep v_E inside the grid.
        ego_acceleration = np.clip(
            ego_acceleration,
            (
                grid_lo[5]
                - v_e
            ) / dt,
            (
                grid_hi[5]
                - v_e
            ) / dt,
        )

        ego_recovery_control = np.array(
            [
                ego_steering_rate,
                ego_acceleration,
            ],
            dtype=float,
        )

        # ====================================================
        # RECOVERY MODE: HUMAN OPTIMIZATION
        # ====================================================

        best_human_control = None
        best_final_distance = np.inf

        for candidate_yaw_rate in human_yaw_candidates:
            for candidate_acceleration in (
                human_acceleration_candidates
            ):
                predicted_joint_state = (
                    current_joint_state.copy()
                )

                # Predict the two vehicles for the requested
                # recovery horizon.
                for prediction_step in range(
                    recovery_prediction_steps
                ):
                    predicted_relative_state = np.asarray(
                        relative_state(
                            jnp.asarray(
                                predicted_joint_state
                            )
                        ),
                        dtype=float,
                    )

                    predicted_delta_e = (
                        predicted_relative_state[4]
                    )

                    predicted_v_e = (
                        predicted_relative_state[5]
                    )

                    predicted_v_h = (
                        predicted_relative_state[3]
                    )

                    # ----------------------------------------
                    # Predicted ego recovery control
                    # ----------------------------------------

                    predicted_ego_steering_rate = np.clip(
                        -predicted_delta_e
                        / RECOVERY_DT,
                        ego_input_min[0],
                        ego_input_max[0],
                    )

                    predicted_ego_acceleration = np.clip(
                        (
                            recovery_reference_speed
                            - predicted_v_e
                        )
                        / RECOVERY_DT,
                        ego_input_min[1],
                        ego_input_max[1],
                    )

                    predicted_ego_acceleration = np.clip(
                        predicted_ego_acceleration,
                        (
                            grid_lo[5]
                            - predicted_v_e
                        )
                        / RECOVERY_DT,
                        (
                            grid_hi[5]
                            - predicted_v_e
                        )
                        / RECOVERY_DT,
                    )

                    predicted_ego_control = np.array(
                        [
                            predicted_ego_steering_rate,
                            predicted_ego_acceleration,
                        ],
                        dtype=float,
                    )

                    # ----------------------------------------
                    # Predicted human control
                    # ----------------------------------------

                    predicted_human_acceleration = np.clip(
                        candidate_acceleration,
                        (
                            grid_lo[3]
                            - predicted_v_h
                        )
                        / RECOVERY_DT,
                        (
                            grid_hi[3]
                            - predicted_v_h
                        )
                        / RECOVERY_DT,
                    )

                    predicted_human_control = np.array(
                        [
                            candidate_yaw_rate,
                            predicted_human_acceleration,
                        ],
                        dtype=float,
                    )

                    predicted_joint_input = np.concatenate(
                        [
                            predicted_ego_control,
                            predicted_human_control,
                        ]
                    )

                    # Euler prediction.
                    predicted_joint_state = (
                        predicted_joint_state
                        + np.asarray(
                            joint_dynamics(
                                jnp.asarray(
                                    predicted_joint_state
                                ),
                                jnp.asarray(
                                    predicted_joint_input
                                ),
                            ),
                            dtype=float,
                        )
                        * RECOVERY_DT
                    )

                    # Numerical protection:
                    # index 3 = delta_E
                    # index 4 = v_E
                    # index 8 = v_H
                    predicted_joint_state[3] = np.clip(
                        predicted_joint_state[3],
                        grid_lo[4],
                        grid_hi[4],
                    )

                    predicted_joint_state[4] = np.clip(
                        predicted_joint_state[4],
                        grid_lo[5],
                        grid_hi[5],
                    )

                    predicted_joint_state[8] = np.clip(
                        predicted_joint_state[8],
                        grid_lo[3],
                        grid_hi[3],
                    )

                # --------------------------------------------
                # Recovery objective
                # --------------------------------------------
                #
                # Evaluate center-to-center distance only at
                # the end of the prediction horizon.

                final_distance = np.linalg.norm(
                    predicted_joint_state[5:7]
                    - predicted_joint_state[0:2]
                )

                candidate_control = np.array(
                    [
                        candidate_yaw_rate,
                        candidate_acceleration,
                    ],
                    dtype=float,
                )

                # Primary criterion:
                # minimum final distance.
                better_distance = (
                    final_distance
                    < best_final_distance - 1e-9
                )

                same_distance = (
                    abs(
                        final_distance
                        - best_final_distance
                    )
                    <= 1e-9
                )

                # Secondary criterion:
                # minimum absolute yaw rate.
                smaller_yaw_rate = (
                    best_human_control is None
                    or abs(candidate_yaw_rate)
                    < abs(
                        best_human_control[0]
                    ) - 1e-9
                )

                same_yaw_rate = (
                    best_human_control is not None
                    and abs(
                        abs(candidate_yaw_rate)
                        - abs(best_human_control[0])
                    )
                    <= 1e-9
                )

                # Tertiary criterion:
                # minimum absolute acceleration.
                smaller_acceleration = (
                    best_human_control is None
                    or abs(candidate_acceleration)
                    < abs(
                        best_human_control[1]
                    ) - 1e-9
                )

                better_tie_break = (
                    smaller_yaw_rate
                    or (
                        same_yaw_rate
                        and smaller_acceleration
                    )
                )

                if (
                    better_distance
                    or (
                        same_distance
                        and better_tie_break
                    )
                ):
                    best_final_distance = final_distance
                    best_human_control = (
                        candidate_control
                    )

        if best_human_control is None:
            raise RuntimeError(
                "The recovery optimization did not select "
                "a valid human control."
            )

        # ====================================================
        # APPLY SELECTED RECOVERY CONTROLS
        # ====================================================

        current_v_h = current_relative_state[3]

        # Clip the selected human acceleration using the real
        # simulation step dt.
        human_acceleration = np.clip(
            best_human_control[1],
            (
                grid_lo[3]
                - current_v_h
            ) / dt,
            (
                grid_hi[3]
                - current_v_h
            ) / dt,
        )

        human_recovery_control = np.array(
            [
                best_human_control[0],
                human_acceleration,
            ],
            dtype=float,
        )

        recovery_joint_input = np.concatenate(
            [
                ego_recovery_control,
                human_recovery_control,
            ]
        )

        # Real simulation propagation.
        next_joint_state = (
            current_joint_state
            + np.asarray(
                joint_dynamics(
                    jnp.asarray(
                        current_joint_state
                    ),
                    jnp.asarray(
                        recovery_joint_input
                    ),
                ),
                dtype=float,
            )
            * dt
        )

        # Numerical protection:
        # joint_state[3] = delta_E
        # joint_state[4] = v_E
        # joint_state[8] = v_H
        next_joint_state[3] = np.clip(
            next_joint_state[3],
            grid_lo[4],
            grid_hi[4],
        )

        next_joint_state[4] = np.clip(
            next_joint_state[4],
            grid_lo[5],
            grid_hi[5],
        )

        next_joint_state[8] = np.clip(
            next_joint_state[8],
            grid_lo[3],
            grid_hi[3],
        )

        joint_states.append(
            next_joint_state
        )

        # The BRT is not defined outside its grid.
        values_history.append(
            np.nan
        )

    return (
        np.asarray(joint_states[:-1]),
        np.asarray(values_history),
    )


def animate_joint_trajectory(
    ego_state,
    human_state,
    dt=1 / 30,
    T=10,
    animation_time_scale_factor=2,
):
    joint_states, values_history = joint_trajectory(
        ego_state,
        human_state,
        dt,
        T,
    )

    xmin = np.min(
        joint_states[:, [0, 5]]
    )

    xmax = np.max(
        joint_states[:, [0, 5]]
    )

    ymin = np.min(
        joint_states[:, [1, 6]]
    )

    ymax = np.max(
        joint_states[:, [1, 6]]
    )

    # Vehicle dimensions [m].
    ego_length = 4.68
    ego_width = 2.20

    human_length = 4.28
    human_width = 1.80

    maximum_half_length = max(
        ego_length,
        human_length,
    ) / 2

    maximum_half_width = max(
        ego_width,
        human_width,
    ) / 2

    fig = plt.figure(
        figsize=(10, 8)
    )

    ax = fig.gca()

    ax.set_xlim(
        xmin - maximum_half_length - 1,
        xmax + maximum_half_length + 1,
    )

    ax.set_ylim(
        ymin - maximum_half_width - 1,
        ymax + maximum_half_width + 1,
    )

    ax.set_aspect(
        "equal",
        adjustable="box",
    )

    # The absolute positions (x_E, y_E) and (x_H, y_H)
    # are assumed to represent the centres of the vehicles.
    ego = ax.add_patch(
        plt.Rectangle(
            (
                -ego_length / 2,
                -ego_width / 2,
            ),
            ego_length,
            ego_width,
            facecolor="tab:blue",
            edgecolor="black",
            linewidth=1.5,
            alpha=0.85,
        )
    )

    human = ax.add_patch(
        plt.Rectangle(
            (
                -human_length / 2,
                -human_width / 2,
            ),
            human_length,
            human_width,
            facecolor="tab:orange",
            edgecolor="black",
            linewidth=1.5,
            alpha=0.85,
        )
    )

    ego_path, = ax.plot(
        [],
        [],
        color="tab:blue",
        linewidth=1,
        alpha=0.7,
        label="Ego",
    )

    human_path, = ax.plot(
        [],
        [],
        color="tab:orange",
        linewidth=1,
        alpha=0.7,
        label="Human",
    )

    value_text = ax.text(
        0.02,
        0.98,
        "",
        transform=ax.transAxes,
        verticalalignment="top",
    )

    ax.grid(True)
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_title(
        "Pursuit-evasion game"
    )
    ax.legend(
        loc="upper right"
    )

    def render_frame(i):
        ego.set_transform(
            transforms.Affine2D()
            .rotate(
                joint_states[i, 2]
            )
            .translate(
                joint_states[i, 0],
                joint_states[i, 1],
            )
            + ax.transData
        )

        human.set_transform(
            transforms.Affine2D()
            .rotate(
                joint_states[i, 7]
            )
            .translate(
                joint_states[i, 5],
                joint_states[i, 6],
            )
            + ax.transData
        )

        ego_path.set_data(
            joint_states[:i + 1, 0],
            joint_states[:i + 1, 1],
        )

        human_path.set_data(
            joint_states[:i + 1, 5],
            joint_states[:i + 1, 6],
        )

        value_text.set_text(
            f"t = {i * dt:.2f} s\n"
            f"V(-3, x_R) = {values_history[i]:.4f}"
        )

        return [
            ego,
            human,
            ego_path,
            human_path,
            value_text,
        ]

    result_animation = HTML(
        animation.FuncAnimation(
            fig,
            render_frame,
            joint_states.shape[0],
            interval=(
                1000
                * dt
                / animation_time_scale_factor
            ),
            blit=True,
        ).to_html5_video()
    )

    plt.close()

    return result_animation



Loaded: /home/alessio-grisorio/TESI/hj_reachability/results/brt/dce.npz
BRT shape: (30, 17, 48, 8, 11, 8)
Gradient shape: (30, 17, 48, 8, 11, 8, 6)
Grid bounds:
[-12.          -8.          -3.14159274   1.          -0.2617994
   1.        ]
[17.          8.          3.14159274 11.          0.2617994  11.        ]


In [2]:
ego_initial_state = np.array(
    [
        0.0,   # x_E
        0.0,   # y_E
        0.0,   # psi_E
        0.0,   # delta_E
        7.0,   # v_E
    ]
)

human_initial_state = np.array(
    [
        9.0,  # x_H
        0.0,   # y_H
        0.0,   # psi_H
        5.0,   # v_H
    ]
)

animate_joint_trajectory(
    ego_initial_state,
    human_initial_state,
    dt=1 / 50,
    T=10,
)


RECOVERY MODE ACTIVATED
Simulation time: 4.560000 s
Simulation step: 228
Outside dimensions:
  x_rel = -12.14099503, grid = [-12.00000000, 17.00000000]
Stored ego reference speed: 7.49001217 m/s

